[Guide Index](README.md)


# 04e SHACL Profiling

The SHACL 1.2 Profiling specification defines how a SHACL shapes graph can be named as a resource and declared a profile of an external specification. Once declared, a successful validation against that shapes graph lets `starlayer.shacl` automatically infer the data's conformance to the specification itself — "this data conforms to this shapes graph" becomes "this data conforms to this published policy," without hand-writing that second claim.


## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections reuse variables from earlier ones.

The data and shapes graphs are built as `StarLayerGraph`, the Oxigraph-backed graph class (see the [Oxigraph wrapper guide](02d-oxigraph-wrapper.ipynb)). `StarShaclSchema` and `validate()` accept it directly, and what they hand back follows it: given `StarLayerGraph` inputs, validation report graphs, `extract_shapes_graph()` and `derive_conforms_to()` results are `StarLayerGraph`s, with pyoxigraph terms and only the prefixes of your input plus the ones their IRIs use (a `StarLayerGraph` starts with no prefix bindings). The bundled ontology graphs take no graph: `starontology.get_ontology_graph()`, `get_conformance_profiles()` and `shacl_profiling.declared_conformance_profile()` return a `StarLayerGraph`.

In [1]:
from rdflib import Namespace, URIRef
from rdflib.namespace import RDF, DCTERMS
from starlayer.graph.oxgraph import StarLayerGraph
from starlayer.shacl import StarShaclSchema, shacl_profiling
from starlayer import starontology

EX = Namespace("http://example.org/")
SH = Namespace("http://www.w3.org/ns/shacl#")

## 1. Packaging SHACL

SHACL 1.2 Profiling provides guidance for three related activities: **packaging SHACL** (identifying a shapes graph as a named resource, and describing its constituent shapes and dependencies - this section), **profiling with SHACL** (describing a profile of a specification and using a SHACL shapes graph to validate data against that profile - sections 2-4), and **profiles of SHACL** (defining a profile of the SHACL language itself, such as the subset of SHACL features a particular implementation supports - sections 5-6).

A few points apply across all three:
- When a data graph is validated against a shapes graph, its conformance to that shapes graph can be recorded.
- A SHACL shapes graph can itself be described as a profile of a specification.
- When a data graph conforms to a shapes graph that profiles a specification, the data graph's conformance to the underlying specification can be inferred.
- A system such as `starlayer.shacl` can define a profile of SHACL that identifies the SHACL features it supports - `starlayer.shacl` is then an implementation of that profile, rather than the profile itself.

Before recording an RDF statement that a data graph conforms to a shapes graph, the shapes graph must be identifiable as a named RDF resource. Packaging provides a way to identify a set of SHACL shapes as a `sh:ShapesGraph`. A resource is designated as a shapes graph using `a sh:ShapesGraph`. Node shapes and property shapes are associated with the shapes graph in three ways:

- `rdfs:isDefinedBy` is used on a node shape or property shape to identify the `sh:ShapesGraph` that defines it. Each shape should have a "home" shapes graph that defines it.
- `rdfs:member` is used on a shapes graph to identify shapes that it uses but does not define. This allows a shapes graph to "borrow" shapes defined by another shapes graph without claiming to define them itself.
- `owl:imports` is used to declare a dependency on another shapes graph in its entirety. This allows a shapes graph to be assembled from components that are independently packaged as shapes graphs.


### 1.1 `starlayer.shacl`'s own packaging support

`starlayer.shacl` supports packaging through two methods on the `StarShaclSchema` object:

- `StarShaclSchema.extract_shapes_graph()` returns a named shapes graph from a `StarShaclSchema` object.
- `StarShaclSchema.validate(..., shapes_graph_resource=...)` validates a data graph using a specified shapes graph. The `shapes_graph_resource` keyword also supports a `starlayer.shacl`-specific convention: when its value does not identify a resource typed as `sh:ShapesGraph`, it is interpreted as the name of the schema's full SHACL graph.


In [2]:
# Alice is a person aged 30.
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person; ex:age 30; ex:name "Alice" .

""", format="turtle")

# Below represents two alternate ways to define the shapes graph.
# rdfs:isDefinedBy links a shape to the sh:ShapesGraph that defines it
# rdfs:member links a sh:ShapesGraph to shapes that it does not define (ex:NameShape below.)
all_shapes = StarLayerGraph()
all_shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

    ex:AdultShapesGraph a sh:ShapesGraph ;
      rdfs:member ex:NameShape.
      
    ex:AdultShape a sh:NodeShape ;
      rdfs:isDefinedBy ex:AdultShapesGraph ;
      sh:targetClass ex:Person ;
      sh:property ex:AdultAgeShape ;
      sh:property ex:NameShape .

    ex:AdultAgeShape a sh:PropertyShape ;
      rdfs:isDefinedBy ex:AdultShapesGraph ;
      sh:path ex:age ;
      sh:datatype xsd:integer ;
      sh:minInclusive 18 .

    ex:ChildShapesGraph a sh:ShapesGraph ;
    rdfs:member ex:NameShape.

    ex:ChildShape a sh:NodeShape ;
      rdfs:isDefinedBy ex:ChildShapesGraph ;
      sh:targetClass ex:Person ;
      sh:property ex:ChildAgeShape ;
      sh:property ex:NameShape .

    ex:ChildAgeShape a sh:PropertyShape ;
      rdfs:isDefinedBy ex:ChildShapesGraph ;
      sh:path ex:age ;
      sh:datatype xsd:integer ;
      sh:maxInclusive 17 .

ex:NameShape a sh:PropertyShape ;     
      sh:path ex:name ;
      sh:datatype xsd:string .


""", format="turtle")

# personSchema includes all of the shapes.  (Adult and Child)
personSchema = StarShaclSchema()
personSchema.shacl_graph= all_shapes

# adultSchema includes only the shapes defined by the EX.AdultShapesGraph
# extract_shapes_graph is used to extract a sh:ShapesGraph from a graph
# and use it to create a new schema object
adultSchema = StarShaclSchema()
adultSchema.shacl_graph= personSchema.extract_shapes_graph(EX.AdultShapesGraph)

# validation fails against the whole shapes graph because both AdultShape and ChildShape are included.
result = personSchema.validate(data_graph=data)
print("whole shapes graph conforms:", result.conforms)


# validation passes against the EX.AdultShapesGraph.
result = adultSchema.validate(data_graph=data)
print("Adult shapes graph conforms:", result.conforms)

# validation fails against the Child shapes graph
# note the followign shows an alternative way to define the shapes graph to be used with validate:
# shapes_graph_resource= here names a real sh:ShapesGraph within personSchema's SHACL shape graph.
# so it's extracted and validated against directly.
result = personSchema.validate(data_graph=data, shapes_graph_resource = EX.ChildShapesGraph)
print("Child shapes graph conforms:", result.conforms)


whole shapes graph conforms: False
Adult shapes graph conforms: True
Child shapes graph conforms: False


/Users/johnclements/Documents/GitHub/hidden-graph/starlayer/packages/starlayer/starlayer/shacl/_adapter.py:210: StarLayerFidelityWarning: The data graph does not preserve literal lexical forms (Oxigraph normalises them on storage: "04"^^xsd:integer becomes "4", "1"^^xsd:boolean becomes "true"), and the shapes use constraints whose result depends on the original lexical form or datatype: sh:datatype. Results may differ from validating the same data with its lexical forms as written. See StarLayer issue #190; upstream oxigraph#526, oxigraph#1989, PR #2010.
  outcome = api.validate(


### 1.2 SHACL packaging and `owl:imports`

A third way a shape can be included in a shapes graph is through `owl:imports` when used to point to another shapes graph. `owl:imports` works similarly to `rdfs:member` above but includes all of the shapes from the imported named graph. This allows for the creation of complex shapes graphs through layered inclusion of simpler shapes graphs.


In [3]:
imports_graph = StarLayerGraph()
imports_graph.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    @prefix owl: <http://www.w3.org/2002/07/owl#> .

    ex:ModuleX a sh:ShapesGraph ;
      owl:imports ex:ModuleY .

    ex:ModuleY a sh:ShapesGraph .
     

    ex:WidgetShape a sh:NodeShape ;
     rdfs:isDefinedBy ex:ModuleY ;
      sh:targetClass ex:Widget ;
      sh:property [ sh:path ex:code ; sh:minCount 1 ] .
""", format="turtle")

importsSchema = StarShaclSchema(shacl_graph=imports_graph)
module_x = importsSchema.extract_shapes_graph(EX.ModuleX)

print("WidgetShape pulled into ModuleX via owl:imports")
print(module_x.serialize(pretty=True, only_used=True))




WidgetShape pulled into ModuleX via owl:imports
@prefix ex: <http://example.org/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix sh: <http://www.w3.org/ns/shacl#> .

ex:ModuleX a sh:ShapesGraph ;
    owl:imports ex:ModuleY .

ex:ModuleY a sh:ShapesGraph .

ex:WidgetShape a sh:NodeShape ;
    rdfs:isDefinedBy ex:ModuleY ;
    sh:property [
        sh:minCount 1 ;
        sh:path ex:code
    ] ;
    sh:targetClass ex:Widget .



## 2. Validation reports and `sh:conformsTo` inference

When a data graph is successfully validated (`conforms=True`) against a shapes graph, the data graph can be recorded as conforming to that shapes graph using `sh:conformsTo`.

When available, the validation report will include identifiers for the data graph (`sh:usedDataGraph`) and shapes graph (`sh:usedShapesGraph`).

The identifier for the data graph is obtained first from the `data_graph_iri=` provided to the validator, if available, or from the `.identifier` property of the data graph. If neither is available, the validation report will not include `sh:usedDataGraph`.

The identifier for the shapes graph is obtained first from the `shapes_graph_resource=` provided to the validator, if available, or from the `.identifier` property of the SHACL graph. If neither is available, the validation report will not include `sh:usedShapesGraph`.


### 2.1 Recording conformance

If both graph identifiers are available and validation succeeds (`conforms=True`), `validate()` adds an `sh:conformsTo` triple to the validation report graph, linking the data graph resource directly to the shapes graph against which it was validated. `report_text` is a human-readable version of the validation output.


In [4]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice ex:age 30 .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:AliceShapesGraph a sh:ShapesGraph ;
      rdfs:member ex:S .
    ex:S a sh:NodeShape ;
      sh:targetNode ex:alice ;
      sh:property [ sh:path ex:age ; sh:datatype xsd:integer ] .
""", format="turtle")

# shapes_graph_resource=ex:AliceShapesGraph names a real sh:ShapesGraph module in the
# bound shacl_graph, so validate() extracts and validates against just that
# module - and its own IRI directly becomes sh:usedShapesGraph, with no
# separate identity to supply; data_graph_iri= is still given explicitly
# for sh:usedDataGraph, since the data graph has no identity of its own.
# ex:data1 (not .../graphs/data1) deliberately shares the ex: namespace
# already bound above, so it prints as ex:data1 rather than minting its
# own separate prefix.
result = StarShaclSchema(shacl_graph=shapes).validate(
    data_graph=data,
    meta_shacl=False,
    data_graph_iri="http://example.org/data1",
    shapes_graph_resource=EX.AliceShapesGraph,
)
print("conforms:", result.conforms)
print("sh:usedDataGraph:", [o.value for o in result.report_graph.objects(None, SH.usedDataGraph)])
print("sh:usedShapesGraph:", [o.value for o in result.report_graph.objects(None, SH.usedShapesGraph)])
# derived automatically by validate() itself, since both IRIs above were present - no separate call needed
print("sh:conformsTo:", [o.value for o in result.report_graph.objects(None, SH.conformsTo)])

print(result.report_graph.serialize(pretty=True, only_used=True))

print(result.report_text)


conforms: True
sh:usedDataGraph: ['http://example.org/data1']
sh:usedShapesGraph: ['http://example.org/AliceShapesGraph']
sh:conformsTo: ['http://example.org/AliceShapesGraph']
@prefix ex: <http://example.org/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .

ex:data1 sh:conformsTo ex:AliceShapesGraph ;
    sh:validationReport [
        a sh:ValidationReport ;
        sh:conformanceDisallows sh:Info, sh:Violation, sh:Warning ;
        sh:conforms true ;
        sh:usedDataGraph ex:data1 ;
        sh:usedShapesGraph ex:AliceShapesGraph
    ] .

Validation Report
Conforms: True

Provenance:
	Used Data Graph: ex:data1
	Used Shapes Graph: ex:AliceShapesGraph
	ex:data1 sh:conformsTo ex:AliceShapesGraph .


## 3. Profiling with SHACL

SHACL Profiles 1.2 outlines how to create a profile of a specification as an RDF graph.

The specification can be any identifiable resource, such as the SHACL 1.2 specification itself or a company expense policy. The profile can then declare that a shapes graph is a profile of that specification.

Once that relationship is declared, validating an expense data graph with a shapes graph implementing that policy not only allows the assertion that the data conforms to the shapes graph (section 2 above), but that the data conforms to the policy itself.

**This second claim is not automatic.** `validate()` always derives `sh:conformsTo` (the shapes graph itself) on its own when `data_graph_iri=`/`shapes_graph_resource=` resolve — but `dcterms:conformsTo` the policy is a separate, deliberate claim about the result, not an intrinsic fact about how validation ran. The same shapes graph could be reused elsewhere without that claim being wanted, so it's made explicitly, after `validate()` returns, by calling `shacl_profiling.derive_conforms_to(result.report_graph, specification_graph=...)` yourself.


In [5]:
PROF = Namespace("http://www.w3.org/ns/dx/prof/")

# A sh:ShapesGraph that defines the expense policy.
# amount must be 500 or less and there must be at least one approver.
expense_shapes = StarLayerGraph()
expense_shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

    ex:ExpenseShapesGraph a sh:ShapesGraph .

    ex:ExpenseShape a sh:NodeShape ;
      rdfs:isDefinedBy ex:ExpenseShapesGraph ;
      sh:targetClass ex:Expense ;
      sh:property ex:AmountShape ;
      sh:property ex:ApproverShape .

    ex:AmountShape a sh:PropertyShape ;
      rdfs:isDefinedBy ex:ExpenseShapesGraph ;
      sh:path ex:amount ;
      sh:datatype xsd:decimal ;
      sh:maxInclusive 500 .

    ex:ApproverShape a sh:PropertyShape ;
      rdfs:isDefinedBy ex:ExpenseShapesGraph ;
      sh:path ex:approver ;
      sh:minCount 1 .
""", format="turtle")

# the specification graph defines the expense policy and
# declares that the ex:ExpenseShapesGraph is an implementation of that policy.
# The policy's identifier is the real URL where it lives  on the comapny intranet.
expense_policy_profile = StarLayerGraph()
expense_policy_profile.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix prof: <http://www.w3.org/ns/dx/prof/> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    @prefix acmePolicy: <https://intranet.example.com/policies/> .

    acmePolicy:expenses
      rdfs:label "Acme Corp Expense Policy" ;
      rdfs:comment "Employee expense claims must not exceed $500 per claim and must name an approver." .

    ex:ExpenseShapesGraph
      prof:isProfileOf acmePolicy:expenses ;
      rdfs:label "Acme Corp Expense Shapes" ;
      rdfs:comment "SHACL implementation of the Acme Corp Expense Policy." .
""", format="turtle")

# specification_graph is not a schema/validate() parameter - validate()
# only ever auto-derives sh:conformsTo on its own. ExpenseSchema is just
# the ordinary shacl_graph-only schema from here on.
ExpenseSchema = StarShaclSchema(shacl_graph=expense_shapes)

# A valid expense: within budget, with an approver.
valid_expense = StarLayerGraph()
valid_expense.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:expense1 a ex:Expense ; ex:amount 450.00 ; ex:approver ex:bob .
""", format="turtle")

result = ExpenseSchema.validate(
    data_graph=valid_expense,
    data_graph_iri="http://example.org/expense1",
    shapes_graph_resource=EX.ExpenseShapesGraph,
)
print("Valid expense (sh:conformsTo only, from validate() itself):")
print(result.report_graph.serialize(pretty=True, only_used=True))

# dcterms:conformsTo the policy is a deliberate, application-level claim -
# call derive_conforms_to() explicitly once validate() has returned. By
# default it returns result.report_graph merged with whatever got
# derived - a self-contained graph, ready to print/serialize on its own,
# with no separate merge step needed.
merged = shacl_profiling.derive_conforms_to(
    result.report_graph,
    specification_graph=expense_policy_profile,
)
print()
print("After calling derive_conforms_to() explicitly:")
print(merged.serialize(pretty=True, only_used=True))

# An over-budget expense - fails ex:ExpenseShape, so sh:conformsTo never
# derives either: the data never conforms to our SHACL rules, so calling
# derive_conforms_to() on this report wouldn't produce anything to chain
# dcterms:conformsTo from.
invalid_expense = StarLayerGraph()
invalid_expense.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:expense2 a ex:Expense ; ex:amount 900.00 ; ex:approver ex:bob .
""", format="turtle")

bad_result = ExpenseSchema.validate(
    data_graph=invalid_expense,
    data_graph_iri="http://example.org/expense2",
    shapes_graph_resource=EX.ExpenseShapesGraph,
)
print()
print("Over-budget expense:")
print(bad_result.report_graph.serialize(pretty=True, only_used=True))


Valid expense (sh:conformsTo only, from validate() itself):
@prefix ex: <http://example.org/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .

ex:expense1 sh:conformsTo ex:ExpenseShapesGraph ;
    sh:validationReport [
        a sh:ValidationReport ;
        sh:conformanceDisallows sh:Info, sh:Violation, sh:Warning ;
        sh:conforms true ;
        sh:usedDataGraph ex:expense1 ;
        sh:usedShapesGraph ex:ExpenseShapesGraph
    ] .


After calling derive_conforms_to() explicitly:
@prefix acmePolicy: <https://intranet.example.com/policies/> .
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix ex: <http://example.org/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .

ex:expense1 dcterms:conformsTo acmePolicy:expenses ;
    sh:conformsTo ex:ExpenseShapesGraph ;
    sh:validationReport [
        a sh:ValidationReport ;
        sh:conformanceDisallows sh:Info, sh:Violation, sh:Warning ;
        sh:conforms true ;
        sh:usedDataGraph ex:expense1 ;
        sh:usedShapesGraph ex:Exp

## 4. Profiles of `starlayer.starontology`

`starlayer` bundles its own set of ontology graphs and related SHACL shapes graphs (see [StarLayer Schemas and Ontologies](05-starlayer-schemas-and-ontologies.ipynb)) — `manch:`, `skos:`, `sqe:`, `spexpr:`, and `srl:`.

**The shapes graphs are profiles of related external specifications:**

- `manch:` — OWL 2 Manchester Syntax
- `skos:` — SKOS Reference
- `sqe:` / `spexpr:` — SPARQL 1.2 Query
- `srl:` — SRL

`starlayer` provides a graph of "conformance profiles" which maps each specification to the SHACL file that validates against it, prepared in line with the SHACL Profiles 1.2 specification. That graph is available via `starontology.get_conformance_profiles()`.

As outlined above, validating a data graph against one of these SHACL graphs produces a `sh:conformsTo` triple linking the data graph to the shapes graph.

Likewise, the conformance profile graph lets `derive_conforms_to()` produce a `dcterms:conformsTo` triple linking the data graph to the specification itself.

The following example demonstrates validating a SKOS data graph against the SKOS SHACL graph and then concluding that the SKOS data graph conforms to the SKOS standard itself.


In [6]:

# SKOS is one of the starlayer supported specifications.
# a small demo SKOS graph is created to be 
# validated by the starlayer SKOS shapes graph
skos_data = StarLayerGraph()
skos_data.parse(data="""
    @prefix skos: <http://www.w3.org/2004/02/skos/core#> .
    @prefix ex: <http://example.org/> .
    ex:scheme1 a skos:ConceptScheme .
    ex:concept1 a skos:Concept ; skos:inScheme ex:scheme1 ; skos:prefLabel "Example"@en .
""", format="turtle")

# the starlayer SKOS SHACL file is obtained.
# note that the skos_shapes resource identifier is returned with the graph
skos_shapes = starontology.get_ontology_graph("skos_shacl")
print("SKOS SHACL Identifier:", skos_shapes.identifier.value)

# the shacl_graph schema object is created from the skos shapes graph
skosSchema = StarShaclSchema(shacl_graph=skos_shapes)


# our demo SKOS data is validated using the skosSchema 
# sh:conformsTo derives automatically when the data graph and shapes graph IRIs are present 
skos_result = skosSchema.validate(
    data_graph=skos_data,
    data_graph_iri="http://example.org/graphs/my-skos-data",
    shapes_graph_resource=skos_shapes.identifier,
)

print()
print("report graph contains sh:conformsTo:")
print(skos_result.report_graph.serialize(pretty=True, only_used=True))

# get_conformance_profiles() returns the starlayer profile graph.
# The profile graph contains a series of specifications that are 
# supported by `starlayer` with a SHACL shapes graph and ontology file.  
profile_graph = starontology.get_conformance_profiles()

# asserting that the data graph dcterms:conformsTo
# a specification is possible when the report graph contains a sh:conformsTo assertion
# and a conformance profile contains the linkage between a 
# shacl graph and a specification.
# derive_conforms_to() performs this inference

# conforms_only=True returns ONLY the sh:conformsTo/dcterms:conformsTo
# triples themselves 
conforms_only = shacl_profiling.derive_conforms_to(
    skos_result.report_graph,
    specification_graph=profile_graph,
    conforms_only=True,
)
print("conforms_only=True:")
print(conforms_only.serialize(pretty=True, only_used=True))

# The default (conforms_only=False) returns report_graph merged
# with the derived conformsTo triples.  
merged = shacl_profiling.derive_conforms_to(
    skos_result.report_graph,
    specification_graph=profile_graph,
)
print()
print("conforms_only=False (the default) - report_graph merged with the derived triples:")
print(merged.serialize(pretty=True, only_used=True))


SKOS SHACL Identifier: https://github.com/hidden-graph/starlayer/blob/main/packages/starlayer/starlayer/starontology/skos_shapes.ttl

report graph contains sh:conformsTo:
@prefix sh: <http://www.w3.org/ns/shacl#> .

<http://example.org/graphs/my-skos-data> sh:conformsTo <https://github.com/hidden-graph/starlayer/blob/main/packages/starlayer/starlayer/starontology/skos_shapes.ttl> ;
    sh:validationReport [
        a sh:ValidationReport ;
        sh:conformanceDisallows sh:Info, sh:Violation, sh:Warning ;
        sh:conforms true ;
        sh:usedDataGraph <http://example.org/graphs/my-skos-data> ;
        sh:usedShapesGraph <https://github.com/hidden-graph/starlayer/blob/main/packages/starlayer/starlayer/starontology/skos_shapes.ttl>
    ] .

conforms_only=True:
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix slp: <https://github.com/hidden-graph/starlayer/ns/profile#> .
@prefix starontology: <https://github.com/hidden-graph/starlayer

The derived triple above — `dcterms:conformsTo` pointing at the real SKOS Reference spec, not `skos_shacl` — only derives because `shapes_graph_resource=skos_shapes.identifier` reuses `skos_shapes`'s own real `.identifier`, the exact same IRI `get_conformance_profiles()`'s own `prof:isProfileOf` triple names as its subject. Passing any other identity for the shapes graph (your own dataset-catalog scheme, say) would still let `sh:conformsTo` derive automatically from `validate()`, but `derive_conforms_to()`'s second rule wouldn't find a matching `prof:isProfileOf` triple to chain from. Calling `derive_conforms_to()` against a non-conforming report, or without `specification_graph=`, derives nothing at all — the second rule has no `sh:conformsTo` triple to chain from if the first rule's own `sh:conforms true` requirement wasn't met.


## 5. Declaring `starlayer.shacl`'s own conformance (§3, "Profiles of SHACL")

SHACL Profiling 1.2 §3 demonstrates how to create a profile *of* a specification — using SHACL's own seven feature-area profiles (Core, SPARQL, Node Expressions, Rules, UI, Profiling, and a Union of all of them) — each with its own IRI.  

1. Each profile is a real, published document declaring itself `prof:isProfileOf <the SHACL 1.2 specification itself>` — that's what creating a profile of a specification looks like.
2. Profiles can, in principle, declare relationships to *other* profiles (§4.4, "Profile Hierarchies").
3. These seven are SHACL's own *official* profiles — minted once, by the SHACL WG itself, and reusable by anyone (including `starlayer.shacl`) wanting to make a conformance claim against one of these profiles.
4. Separately, §5.5 shows that a SHACL *shapes graph* can declare itself `prof:isProfileOf` a specification, and `derive_conforms_to()`'s second rule then lets data that conforms to that shapes graph transitively conform to the specification too — the same mechanism Sections 3 and 4 already demonstrated for an expense policy and the real SKOS Reference.

`starlayer.shacl` declares its own conformance to five of the seven official profiles via `declared_conformance_profile()` — a bundled RDF document asserting `dcterms:conformsTo` on `starlayer.shacl` itself against each profile IRI it genuinely implements. It also claims the SHACL 1.2 UI specification itself (`tr:shacl12-ui`) within documented scope limits, because the widget-selection algorithm is implemented (see the [SHACL UI guide](04d-shacl-ui.ipynb)). The UI *Profile* is not claimed, and a comment on the resource says why.


In [7]:
from starlayer.shacl import shacl_profiling
from starlayer import conformance
from rdflib.namespace import RDFS

shacl_declaration = shacl_profiling.declared_conformance_profile()
shacl_declaration.bind("profile", Namespace("http://www.w3.org/ns/shacl/profile/"))
shacl_declaration.bind("conformance", conformance.CONFORMANCE)
shacl_declaration.remove((None, RDFS.seeAlso, None))
print(shacl_declaration.serialize(format="turtle", pretty=True, only_used=True))


@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix pkg: <https://github.com/hidden-graph/starlayer/tree/main/packages/starlayer/starlayer/> .
@prefix profile: <http://www.w3.org/ns/shacl/profile/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix tr: <https://www.w3.org/TR/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

pkg:shacl dcterms:conformsTo profile:core, profile:node-expr, profile:profiling, profile:rules, profile:sparql, tr:shacl12-ui ;
    dcterms:modified "2026-10-17"^^xsd:date ;
    rdfs:comment "Not claimed: the UI Profile. SHACL 1.2 Profiling defines it only as a subset of the SHACL Core vocabulary chosen for ease of UI implementation, lists no members and states no conformance requirements (Appendix A.3 is a TODO), so there is nothing to claim or exclude yet. The SHACL 1.2 UI widget-selection algorithm is implemented separately - see docs/shacl/shacl12-gap-matrix.md."@en,
        "SHACL 1.2 User Interfaces is claimed within documented scope li

`pkg:shacl`'s `rdfs:comment` values state what it deliberately doesn't claim and the scope limits on what it does — real RDF data on the resource itself, not just source-file prose.


Point 4 above applies here directly: `starlayer`'s own SHACL meta-shapes graph (shapes-about-shapes, registered as `"shacl_shacl"`) itself declares `prof:isProfileOf` each public profile it represents — the same five `starlayer.shacl` already claims above — right in `starontology`'s own bundled `get_conformance_profiles()` document, the same place every other registered pair's relationship lives. Validating any shapes graph against `shacl_shacl` and calling `derive_conforms_to()` then derives a real `dcterms:conformsTo` claim against those public profiles directly, the same mechanism Sections 3 and 4 used for the expense policy and the real SKOS Reference.


In [8]:
shacl_shacl = starontology.get_ontology_graph("shacl_shacl")
skos_shacl_graph = starontology.get_ontology_graph("skos_shacl")

# skos_shacl (the data graph here) is validated against shacl_shacl (the
# shapes graph) - shapes_graph_resource= tags the report with shacl_shacl's
# own real resource IRI, the same one get_conformance_profiles() below
# asserts prof:isProfileOf on, so derive_conforms_to()'s second rule can
# actually match it.
meta_result = StarShaclSchema(shacl_graph=shacl_shacl).validate(
    data_graph=skos_shacl_graph,
    data_graph_iri=skos_shacl_graph.identifier,
    shapes_graph_resource=starontology.get_resource_iri("shacl_shacl"),
    meta_shacl=False,
    ignore_imports=True,  # shacl_shacl declares owl:imports of the SHACL vocabulary, which is not loaded here
)

# derive_conforms_to() chains sh:conformsTo (skos_shacl -> shacl_shacl)
# into dcterms:conformsTo (skos_shacl -> each public SHACL profile
# shacl_shacl represents) - the default return is report_graph merged
# with whatever got derived, ready to print on its own. The full,
# unfiltered mapping graph works fine here - derive_conforms_to()'s own
# WHERE clause already filters by the exact shapes-graph IRI.
merged = shacl_profiling.derive_conforms_to(
    meta_result.report_graph,
    specification_graph=starontology.get_conformance_profiles(),
)
print(merged.serialize(pretty=True, only_used=True))

@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix profile: <http://www.w3.org/ns/shacl/profile/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .
@prefix slp: <https://github.com/hidden-graph/starlayer/ns/profile#> .
@prefix starontology: <https://github.com/hidden-graph/starlayer/blob/main/packages/starlayer/starlayer/starontology/> .

starontology:skos_shapes.ttl dcterms:conformsTo profile:core, profile:node-expr, profile:profiling, profile:rules, profile:sparql, slp:Shacl ;
    sh:conformsTo starontology:shacl12-shacl.ttl ;
    sh:validationReport [
        a sh:ValidationReport ;
        sh:conformanceDisallows sh:Info, sh:Violation, sh:Warning ;
        sh:conforms true ;
        sh:usedDataGraph starontology:skos_shapes.ttl ;
        sh:usedShapesGraph starontology:shacl12-shacl.ttl
    ] .



/Users/johnclements/Documents/GitHub/hidden-graph/starlayer/packages/starlayer/starlayer/shacl/_adapter.py:210: StarLayerFidelityWarning: The data graph does not preserve literal lexical forms (Oxigraph normalises them on storage: "04"^^xsd:integer becomes "4", "1"^^xsd:boolean becomes "true"), and the shapes use constraints whose result depends on the original lexical form or datatype: sh:closed, sh:datatype, sh:pattern. Results may differ from validating the same data with its lexical forms as written. See StarLayer issue #190; upstream oxigraph#526, oxigraph#1989, PR #2010.
  outcome = api.validate(


## 6. Profiles of `starlayer` itself

This is not yet implemented as code - documented here as the direction this pattern would extend to, not as something callable today.

Extending the same pattern one level further: `starlayer` implements a documented subset of RDF 1.2, SPARQL 1.2, and SHACL 1.2. Following §3's own two-step relationship (a profile is a *specification*; an implementation *conforms to* it, rather than being a profile itself) — the actual profile isn't `starlayer` the software, it's `starlayer`'s own documented scope: the gap-analysis documents, the scope-reduction notes already scattered through each subpackage's own `CLAUDE.md` (SRL's documented grammar reductions, the RDF 1.2 interop gaps, etc.). `starlayer` would then *conform to* that profile — the same relationship a SHACL validator has with the Core Profile.

Breaking this down by subpackage:
- `starlayer.graph` would be a profile of RDF 1.2 (Concepts/Schema/Turtle/N-Triples/XML/Canonicalization), SPARQL 1.2 Protocol, and the individually-supported Entailment Regimes.
- `starlayer.sparql` would be a profile of SPARQL 1.2 Query/Update/RL.
- `starlayer.shacl` would be a profile of SHACL 1.2 (formalized above in section 5).
- **`starlayer` specifics** — vocabulary/tooling with no external spec to be a profile *of* at all: `manch:`/`sqe:`/`spexpr:`/`srl:` (`starlayer`'s own invented syntax-tree encodings), the `shnex:` node-expression extensions. These don't fit the "profile of X" framing — there's no `prof:isProfileOf` target for them — and would need some other treatment, not a forced claim.
- The **whole-library roll-up** ("the `starlayer` profile" as the union of the above) has no ready-made formal mechanism to lean on. Even SHACL's own "Union Profile" is prose-only in the spec itself ("the conceptual whole... covers all aspects"), not a derivable triple anywhere in dx-prof — a precedent for staying prose-only here too, though reaching outside dx-prof entirely (e.g. Dublin Core's `dcterms:hasPart`) for something actually derivable is also an option.

What's still undecided: whether each subpackage should get its own real `prof:Profile` resource (mirroring SHACL's own Core/SPARQL/etc.), replacing the current plain `dcterms:conformsTo <raw TR document>` claim in `conformance.declared_conformance()`; what, if anything, the roll-up/union should assert beyond prose; and where the `starlayer`-specific vocabulary (no external spec) gets documented, if not as a profile claim at all.
